### Описание датасета

Набор данных содержит информацию о транзакциях по кредитным картам, совершенных европейскими держателями карт в сентябре 2013 года. В этом наборе данных представлены транзакции за два дня, из которых 492 являются мошенническими из 284 807 транзакций. Набор данных сильно несбалансирован: на положительный класс (мошенничество) приходится 0,172 % всех транзакций. Он содержит только числовые входные переменные, полученные в результате преобразования методом главных компонент. К сожалению, из соображений конфиденциальности мы не можем предоставить исходные признаки и дополнительную информацию о данных. Признаки V1, V2, V28 — это главные компоненты, полученные с помощью метода главных компонент. Единственные признаки, которые не были преобразованы с помощью метода главных компонент, — это «Время» и «Сумма». Признак «Время» содержит количество секунд, прошедших между каждой транзакцией и первой транзакцией в наборе данных. Параметр 'Amount' — это сумма транзакции. Этот параметр можно использовать для обучения с учетом затрат. Параметр 'Class' — это переменная ответа, которая принимает значение 1 в случае мошенничества и 0 в остальных случаях.

In [1]:
from pathlib import Path
from scipy.io import arff

import pandas as pd


data_path = Path("../data/external/dataset")
data, meta = arff.loadarff(data_path)

df = pd.DataFrame(data)

if "Class" in df.columns:
    df["Class"] = df["Class"].astype(int)

df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


In [2]:
missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()

print(f"Размер датасета: {df.shape[0]:,} строк, {df.shape[1]} признак")
print(f"Общее число пропусков: {int(missing.sum())}")
print(f"Число дубликатов: {int(duplicates)}")

class_counts = df["Class"].value_counts().sort_index()
class_share = (class_counts / len(df) * 100).round(4)

print("\nРаспределение классов (по типу операции: 1 - мошенник, 0 - нет):")
display(pd.DataFrame({"count": class_counts, "share_%": class_share}))

major = class_counts.max()
minor = class_counts.min()
print(f"Imbalance ratio (major/minor): {major/minor:.2f}:1")

print("\nКлючевая статистика Amount и Time:")
display(df[["Amount", "Time"]].describe().T)

Размер датасета: 284,807 строк, 31 признак
Общее число пропусков: 0
Число дубликатов: 1081

Распределение классов (по типу операции: 1 - мошенник, 0 - нет):


,count,share_%
Class,,
0,284315,99.8273
1,492,0.1727


Imbalance ratio (major/minor): 577.88:1

Ключевая статистика Amount и Time:


,count,mean,std,min,25%,50%,75%,max
Amount,284807.0,88.349619,250.120109,0.0,5.6,22.0,77.165,25691.16
Time,284807.0,94813.859575,47488.145955,0.0,54201.5,84692.0,139320.500,172792.00


In [4]:
%matplotlib inline
import matplotlib
matplotlib.use("agg")

if not hasattr(pd.DataFrame, "applymap"):
    pd.DataFrame.applymap = pd.DataFrame.map

from autoviz import AutoViz_Class
import os


save_dir = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", "reports", "figures"))
os.makedirs(save_dir, exist_ok=True)

AV = AutoViz_Class()
dft = AV.AutoViz(
    filename="",
    sep=",",
    depVar="Class",
    dfte=df,
    header=0,
    verbose=2,
    lowess=False,
    chart_format="png",
    max_rows_analyzed=150000,
    max_cols_analyzed=31,
    save_plot_dir=save_dir,
)

    Since nrows is smaller than dataset, loading random sample of 150000 rows into pandas...
Shape of your Data Set loaded: (150000, 31)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
  Printing up to 30 columns (max) in each category:
    Numeric Columns : ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']
    Integer-Categorical Columns: []
    String-Categorical Columns: []
    Factor-Categorical Columns: []
    String-Boolean Columns: []
    Numeric-Boolean Columns: []
    Discrete String Columns: []
    NLP text Columns: []
    Date Time Columns: []
    ID Columns: []
    Columns

,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
Time,float64,0.000000,NA,1.000000,172788.000000,No issue
V1,float64,0.000000,NA,-46.855047,2.454930,Column has 3684 outliers greater than upper bound (4.67) or lower than lower bound(-4.28). Cap them or remove them.
V2,float64,0.000000,NA,-63.344698,19.167239,Column has 7096 outliers greater than upper bound (2.89) or lower than lower bound(-2.70). Cap them or remove them.
V3,float64,0.000000,NA,-33.680984,4.187811,Column has 1755 outliers greater than upper bound (3.91) or lower than lower bound(-3.77). Cap them or remove them.
V4,float64,0.000000,NA,-5.683171,16.715537,Column has 5795 outliers greater than upper bound (3.12) or lower than lower bound(-3.23). Cap them or remove them.
V5,float64,0.000000,NA,-40.427726,32.911462,Column has 6382 outliers greater than upper bound (2.56) or lower than lower bound(-2.65). Cap them or remove them.
V6,float64,0.000000,NA,-21.929312,23.917837,Column has 12072 outliers greater than upper bound (2.13) or lower than lower bound(-2.51). Cap them or remove them.
V7,float64,0.000000,NA,-37.060311,44.054461,Column has 4630 outliers greater than upper bound (2.26) or lower than lower bound(-2.24). Cap them or remove them.
V8,float64,0.000000,NA,-73.216718,20.007208,Column has 12682 outliers greater than upper bound (1.13) or lower than lower bound(-1.01). Cap them or remove them.
V9,float64,0.000000,NA,-11.126624,10.392889,Column has 4211 outliers greater than upper bound (2.47) or lower than lower bound(-2.52). Cap them or remove them.


Total Number of Scatter Plots = 465
All Plots are saved in /Users/stepa/Study/master/mlops/reports/figures/Class
Time to run AutoViz = 99 seconds 


## Выводы EDA



1. **Размер датасета:** 284 807 транзакций, 31 признак (Time, V1–V28, Amount, Class).

2. **Пропуски:** отсутствуют. **Дубликаты:** 1 081 (≈ 0.38 %) — некритично, могут быть реальными повторными транзакциями.

3. **Дисбаланс классов:** Class=0 — 284 315 (99.83 %), Class=1 — 492 (0.17 %). Соотношение ≈ 578:1. Это **экстремальный дисбаланс**.

4. **Amount:** сильно скошено вправо. Мошеннические транзакции в среднем небольшие по сумме.

5. **Корреляции с Class:** наиболее связаны V14, V17, V12, V10 (отрицательно) и V11, V4 (положительно). Amount и Time имеют слабую корреляцию.
